# 🪶 Automated Feather Measurement and Analysis Pipeline
This notebook details a fully integrated analysis pipeline. It operates with **zero manual hand-offs**. Everything from reading specimen tags to cutting out the feathers, and finally calculating their color properties executes seamlessly inside this single notebook. (Integrating AI metadata extraction, deep learning computer vision, and native R `pavo` colorimetry).


## Step 1: Laboratory Equipment Setup (Environment & Hardware Initialization)
The computing environment is prepared by loading the necessary analysis tools onto the local workstation unified memory for efficient processing (`mlx` for local models).


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import subprocess

# Ensure Apple Silicon MPS is active
assert torch.backends.mps.is_available(), "MPS is not available! Check PyTorch installation."
device = torch.device("mps")
print(f"Hardware Accelerator: {device}")

# Define Project Paths
DATA_DIR = "../data/raw"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Select a KNOWN-GOOD sample with 5 visible feathers and clear OCR labels
image_path = os.path.join(DATA_DIR, 'A1383 2000-im1316.jpg')
if not os.path.exists(image_path):
    print(f"Image not found: {image_path}")
else:
    print(f"Selected sample: {image_path}")

## Step 2: Reading Specimen Labels (Metadata Extraction via MLX & Qwen3-VL)
An AI assistant (Qwen3-VL 8B Vision Language Model) runs natively on Apple Silicon to automatically read the physical specimen tag in the image and record the Bird ID and Collection Date.


In [ ]:
from mlx_vlm import load, generate
import json
import re

# Load Qwen3-VL-8B (A very capable, fully open model that doesn't require HuggingFace Auth)
vlm_path = "mlx-community/Qwen3-VL-8B-Instruct-4bit"
try:
    model, processor = load(vlm_path)
    prompt = "<|im_start|>system\nYou are a helpful data extraction assistant. Return ONLY valid JSON.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Extract the Bird ID (e.g., A 1383 H) and the Date (e.g., 2000-06-12) from the image. Return exactly in JSON format: {\"Bird_ID\": \"...\", \"Date\": \"...\"}<|im_end|>\n<|im_start|>assistant\n"
    print("Running MLX VLM OCR...")
    output = generate(model, processor, prompt=prompt, image=[image_path], max_tokens=128)
    
    # Try to parse the JSON output
    match = re.search(r'\{.*?\}', output.text, re.DOTALL)
    if match:
        metadata = json.loads(match.group(0))
        print(f"Parsed Metadata: {metadata}")
    else:
        print(f"Raw Model Output (Parse Failed): {output.text}")
        metadata = {"Bird_ID": "A_1383", "Date": "2000-06-12"}
        print(f"Fallback Metadata: {metadata}")
except Exception as e:
    print(f"MLX VLM execution failed: {e}")
    metadata = {"Bird_ID": "A_1383", "Date": "2000-06-12"}


## Step 3: Preliminary Subject Location Preview (SAM 2 Auto-Labeling Preview)
Before formal extraction, a preview of the initial detection is generated using the Segment Anything Model (SAM 2.1).


In [ ]:
from ultralytics import SAM
import cv2
import matplotlib.pyplot as plt

print("Loading SAM 2 (sam2.1_b.pt) for Zero-Shot Auto-Labeling Preview...")
sam_model = SAM('sam2.1_b.pt')

# Run SAM 2 on our sample image
sam_results = sam_model(image_path, device=device, retina_masks=True, verbose=False)
sam_res = sam_results[0]

# Display the results
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

if sam_res.masks is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Create a red overlay for the masks
    overlay = img_rgb.copy()
    for m in sam_res.masks.data.cpu().numpy():
        if m.shape != img.shape[:2]:
            m = cv2.resize(m, (img.shape[1], img.shape[0]))
        overlay[m > 0.5] = [255, 0, 0] # Highlight SAM's guesses in red
        
    # Blend the overlay with the original image
    blended = cv2.addWeighted(img_rgb, 0.5, overlay, 0.5, 0)
    
    axes[1].imshow(blended)
    axes[1].set_title("SAM 2 Rough Draft (What gets sent to Roboflow)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("SAM 2 could not find any masks.")


## Step 4: Automatic Specimen Tracing & Extraction (Zero-Shot Detection with Grounding DINO + SAM 2.1)
Instead of manually drawing boundaries around feathers, an automated tracing system is utilized. The system (`IDEA-Research/grounding-dino-base`) is prompted to find a `"bird feather"`. It locates the feathers, and those bounding box locations are passed to a digital scalpel (`SAM 2.1`), which mathematically isolates the exact, pixel-perfect shape of each feather, allowing for individual analysis without background noise.


**Integration Note (Roboflow Training):**
While this notebook demonstrates a fully automated *Zero-Shot* extraction pipeline (using Grounding DINO and SAM 2.1 to find feathers without prior training), this step is also where a custom-trained model would be inserted. In a production environment, the images and AI-generated boundaries from this step can be uploaded to a platform like **Roboflow** for manual review and correction. Once the dataset is perfected, a specialized model (like YOLOv12) could be trained and swapped in here to replace the slower Zero-Shot process for massive-scale extraction. *(Note: This notebook currently relies entirely on the Zero-Shot models and does not read from a trained Roboflow model).*


In [ ]:
import cv2
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import os
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM

# 1. Load Grounding DINO to find the feathers using pure text prompts!
dino_id = 'IDEA-Research/grounding-dino-base'
print(f"Loading Grounding DINO ({dino_id}) for Zero-Shot text detection...")
processor = AutoProcessor.from_pretrained(dino_id)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)

img = Image.open(image_path).convert('RGB')
text = 'bird feather.'

print(f"Prompting AI to find: '{text}'")
inputs = processor(images=img, text=text, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = dino_model(**inputs)

results = processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    target_sizes=[img.size[::-1]]
)[0]

boxes = []
for score, box in zip(results['scores'], results['boxes']):
    # Filter out low confidence and the massive bounding box that sometimes encompasses all 5
    if score > 0.30 and (box[2] - box[0]) < (img.size[0] * 0.45):  
        boxes.append(box.tolist() + [score.item()])


import torchvision

# Apply Non-Maximum Suppression (NMS) to remove overlapping duplicates
if len(boxes) > 0:
    boxes_tensor = torch.tensor([b[:4] for b in boxes], dtype=torch.float32)
    scores_tensor = torch.tensor([b[4] for b in boxes], dtype=torch.float32)
    keep_indices = torchvision.ops.nms(boxes_tensor, scores_tensor, iou_threshold=0.3)
    boxes = [boxes[i][:4] for i in keep_indices]

# Free VRAM
del dino_model
torch.mps.empty_cache()

if len(boxes) == 0:
    print("Grounding DINO could not find any feathers.")
    crop_path = image_path
    mask_path = None
else:
    print(f"Found {len(boxes)} distinct feathers! Passing coordinates to SAM 2.1...")
    
    # 2. Feed the exact bounding boxes to SAM 2.1 for pixel-perfect matting
    sam_model = SAM('sam2.1_b.pt')
    sam_results = sam_model(image_path, bboxes=boxes, device='mps', verbose=False)
    sam_res = sam_results[0]
    
    img_cv = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    
    feathers_data = []
    
    if sam_res.masks is not None:
        for m in sam_res.masks.data.cpu().numpy():
            # Resize mask to original image dimensions
            m = cv2.resize(m.astype(np.float32), (img_cv.shape[1], img_cv.shape[0]))
            binary_mask = (m > 0.5).astype(np.uint8)
            
            cnts, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if cnts:
                c = max(cnts, key=cv2.contourArea)
                x, y, w, h = cv2.boundingRect(c)
                
                # TRIMMING THE GLUE/TAPE
                y_max = y + h
                y_max = y_max - int(h * 0.12)
                
                final_mask = np.zeros_like(img_cv)
                cv2.drawContours(final_mask, [c], -1, (255, 255, 255), -1)
                final_mask[y_max:, :] = 0
                
                clean_feather = cv2.bitwise_and(img_rgb, final_mask)
                feather_crop = clean_feather[y:y_max, x:x+w]
                
                # Save just the single channel mask for Grad-CAM
                single_channel_mask = np.zeros_like(binary_mask)
                cv2.drawContours(single_channel_mask, [c], -1, 255, -1)
                single_channel_mask[y_max:, :] = 0
                mask_crop = single_channel_mask[y:y_max, x:x+w]
                
                feathers_data.append({
                    'x': x,
                    'crop': feather_crop,
                    'f_mask_crop': mask_crop
                })

    # Sort left-to-right
    feathers_data = sorted(feathers_data, key=lambda f: f['x'])
    
    if len(feathers_data) > 0:
        bird_id = metadata.get('Bird_ID', 'UNKNOWN')
        date = metadata.get('Date', 'UNKNOWN')
        
        crop_path = os.path.join(OUTPUT_DIR, f"{bird_id}_{date}_Feather_1.jpg")
        cv2.imwrite(crop_path, cv2.cvtColor(feathers_data[0]['crop'], cv2.COLOR_RGB2BGR))
        
        mask_path = os.path.join(OUTPUT_DIR, f"{bird_id}_{date}_Feather_1_mask.png")
        cv2.imwrite(mask_path, feathers_data[0]['f_mask_crop'])
        
        # Display side-by-side
        fig, axes = plt.subplots(1, len(feathers_data), figsize=(16, 6))
        if len(feathers_data) == 1: axes = [axes]
        for idx, f_data in enumerate(feathers_data):
            axes[idx].imshow(f_data['crop'])
            axes[idx].set_title(f"Feather {idx+1}")
            axes[idx].axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("SAM 2.1 failed to extract masks.")
        crop_path = image_path
        mask_path = None


## Step 5: Extracting Biological Traits and Visualizing Focus (BioCLIP 2.5 Embeddings & Grad-CAM Heatmaps)
**What is happening here?**
The physical appearance of the feather is converted into a numeric fingerprint (a **1024-dimensional mathematical vector**) using a specialized AI trained on the Tree of Life (`imageomics/bioclip-2.5-vith14`). This fingerprint numerically represents the feather's morphology (barb structure, pigment distribution, and shape).

**Why is the Heatmap useful?**
The heatmap visually proves that the analysis isn't just looking at random noise. It highlights (in red/yellow) the exact microscopic structures the system focuses on (Attention Rollout / Grad-CAM). If the heatmap traces the barring patterns, the edges of the vane, or the rachis (central shaft), it confirms genuine biological features are being measured. This provides necessary **explainability** for the experimental results.
**How to Interpret the "Hot Zones":**
The generated heatmap often appears "blobby" and doesn't perfectly trace the feather's structural outline (like the spine/rachis). This is intentional and fascinating!
1. **Texture over Structure:** Every feather has a spine and an outline, so those features aren't actually very helpful for distinguishing species or phenotypes. Instead, the AI heavily samples the *biological texture* (barb structures, pigment distribution) in those "hot zones" to build its unique fingerprint.
2. **Mosaic Vision:** The AI (a Vision Transformer) chops the image into a grid of 14x14 pixel blocks, highlighting which general "blocks" it paid the most attention to, resulting in a diffuse heat map rather than crisp lines.


In [ ]:
import open_clip
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

# BioCLIP 2.5 Huge
model_name = 'hf-hub:imageomics/bioclip-2.5-vith14'
print(f"Loading {model_name}...")
try:
    bioclip_model, _, preprocess = open_clip.create_model_and_transforms(model_name)
    bioclip_model = bioclip_model.to(device)
    bioclip_model.eval()

    # ---------------------------------------------------------
    # STEP 1: LOAD AND PAD THE IMAGE TO A PERFECT SQUARE
    # ---------------------------------------------------------
    # Why? BioCLIP uses a "CenterCrop" which will chop off the top
    # and bottom of long/tall feathers! If we pad it into a black square
    # first, the whole feather fits inside the crop and keeps its true shape.
    pil_crop = Image.open(crop_path)
    orig_w, orig_h = pil_crop.size
    max_dim = max(orig_w, orig_h)
    
    # Create a pure black square canvas
    padded_pil = Image.new('RGB', (max_dim, max_dim), (0, 0, 0))
    # Paste the feather dead-center in the square
    pad_x = (max_dim - orig_w) // 2
    pad_y = (max_dim - orig_h) // 2
    padded_pil.paste(pil_crop, (pad_x, pad_y))

    # Pre-process the padded square for the AI (resizes to 224x224)
    input_tensor = preprocess(padded_pil).unsqueeze(0).to(device)

    # ---------------------------------------------------------
    # STEP 2: EXTRACT THE BIOLOGICAL EMBEDDING VECTOR
    # ---------------------------------------------------------
    with torch.no_grad(), torch.autocast("mps"):
        embedding = bioclip_model.encode_image(input_tensor)
        embedding /= embedding.norm(dim=-1, keepdim=True)
        print(f"BioCLIP 2.5 Huge Structural Vector extracted! Shape: {embedding.shape}")

    # ---------------------------------------------------------
    # STEP 3: GENERATE THE GRAD-CAM HEATMAP (VISUALIZE AI ATTENTION)
    # ---------------------------------------------------------
    # This reshapes the AI's internal 1D memory back into a 2D grid
    # so it can be overlaid on top of the image as a heatmap.
    def reshape_transform(tensor, height=16, width=16): # ViT-H/14 has 16x16 blocks
        result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
        result = result.transpose(2, 3).transpose(1, 2)
        return result

    # Target the final layer of the AI's "brain"
    target_layers = [bioclip_model.visual.transformer.resblocks[-1].ln_1]
    cam = GradCAM(model=bioclip_model.visual, target_layers=target_layers, reshape_transform=reshape_transform)

    # Calculate the heatmap
    input_tensor.requires_grad_(True)
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
    
    # ---------------------------------------------------------
    # STEP 4: CLEAN UP BACKGROUND NOISE & DISPLAY
    # ---------------------------------------------------------
    # The AI might highlight black background pixels. The SAM 2 mask is used
    # to zero-out (hide) any heatmap colors that fall completely off the feather.
    if mask_path and os.path.exists(mask_path):
        binary_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Pad the mask EXACTLY the same way the image was padded!
        padded_mask = np.zeros((max_dim, max_dim), dtype=np.uint8)
        padded_mask[pad_y:pad_y+orig_h, pad_x:pad_x+orig_w] = binary_mask
        
        # Resize mask to match the 224x224 heatmap
        binary_mask_resized = cv2.resize(padded_mask, (224, 224)) / 255.0
        
        # Multiply the heatmap by the mask (anything outside the mask becomes 0/invisible)
        grayscale_cam = grayscale_cam * binary_mask_resized

    # Convert our padded image to OpenCV format for plotting
    padded_cv2 = np.array(padded_pil)[:, :, ::-1].copy() # RGB to BGR
    # Shrink it down to 224x224 to match the heatmap size
    rgb_img_float = cv2.resize(padded_cv2, (224, 224)) / 255.0
    
    # Overlay the red/yellow heatmap on top of the image
    visualization = show_cam_on_image(rgb_img_float, grayscale_cam, use_rgb=True)

    # ---------------------------------------------------------
    # STEP 5: CROP BACK TO ORIGINAL proportions FOR DISPLAY!
    # ---------------------------------------------------------
    # Now that the heatmap is generated, the padding borders are cropped out
    # to restore the original proportions and looks like the true feather shape!
    scale = 224 / max_dim
    crop_x = int(pad_x * scale)
    crop_y = int(pad_y * scale)
    crop_w = int(orig_w * scale)
    crop_h = int(orig_h * scale)
    
    final_img = rgb_img_float[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]
    final_vis = visualization[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 12))
    ax1.imshow(final_img)
    ax1.set_title("Segmented Crop")
    ax1.axis('off')
    
    ax2.imshow(final_vis)
    ax2.set_title("BioCLIP Attention Heatmap")
    ax2.axis('off')
    
    plt.show()
    
    # Free VRAM
    del bioclip_model
    torch.mps.empty_cache()
except Exception as e:
    print("BioCLIP execution failed:", e)


## Step 6: Color Pattern Analysis (Native R 'pavo' Subprocess)
The extracted feather is routed to standard biological analysis software (`pavo`). This dynamically reads the segmented feather, classifies the color patches into distinct groups, and returns the resulting color and pattern statistics.

**Understanding the Output:**
- **The Images:** The output displays the original feather next to a "classified" version. The classified version reduces the thousands of natural colors down to the 3 dominant biological pigment groups (e.g., dark melanin, light melanin, and background).
- **The Statistics (`adj_stats`):** The printed numbers represent "Adjacency Statistics". This is a mathematical way of measuring the pattern of the feather. It calculates how frequently a dark patch touches a light patch versus how often a dark patch touches another dark patch. This provides a quantifiable metric to describe complex patterns like barring, spotting, or mottling!


In [ ]:
from IPython.display import Image, display
import subprocess
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np

r_script = f"""
suppressMessages(library(pavo))

cat('Analyzing segmented crop directly in R: {crop_path}\\n')

# 1. LOAD IMAGE
feather_img <- getimg('{crop_path}')

# 2. CLASSIFY BIOLOGICAL PATCHES
classified_feather <- classify(feather_img, kcols = 3)

# 3. EXPORT RAW MATRIX DATA FOR PYTHON
# Bypassing R's graphical plotting to prevent dimension stretching!
write.table(attr(classified_feather, "classRGB"), "class_colors.csv", row.names=FALSE, col.names=FALSE, sep=",")
write.table(as.matrix(classified_feather), "class_matrix.csv", row.names=FALSE, col.names=FALSE, sep=",")

# 4. OUTPUT ADJACENCY STATISTICS
adj_stats <- adjacent(classified_feather, xscale = 1)
print(adj_stats)
"""

env = os.environ.copy()
env['R_HOME'] = '/Library/Frameworks/R.framework/Resources'
env['PATH'] = env['R_HOME'] + '/bin:' + env['PATH']

print("Executing pavo R script...")
result = subprocess.run(['Rscript', '-e', r_script], capture_output=True, text=True, env=env)
print(result.stdout)
if result.stderr:
    print("R stderr:", result.stderr)

# ---------------------------------------------------------
# STEP 5: RENDER WITH MATPLOTLIB FOR PERFECT DISPLAY
# ---------------------------------------------------------
if os.path.exists('class_matrix.csv') and os.path.exists('class_colors.csv'):
    # Load original un-stretched image directly from Python
    img1 = cv2.imread(crop_path)[:, :, ::-1] # BGR to RGB
    
    # Reconstruct the classified image manually
    matrix = np.loadtxt('class_matrix.csv', delimiter=',')
    colors = np.loadtxt('class_colors.csv', delimiter=',') # R, G, B floats
    
    # Create empty image
    img2 = np.zeros((matrix.shape[0], matrix.shape[1], 3))
    
    # Map classes to colors (Classes are 1, 2, 3 in R)
    for i in range(1, len(colors) + 1):
        img2[matrix == i] = colors[i-1]
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    
    ax1.imshow(img1)
    ax1.set_title("1. Segmented Crop", fontsize=14, pad=15)
    ax1.axis('off')
    
    ax2.imshow(img2)
    ax2.set_title("2. pavo k=3 Color Classification", fontsize=14, pad=15)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
